# RestoreAI — Training Denoising & Super-Resolution Models

This notebook trains **two small, real deep-learning models** on the **BSDS300** dataset you uploaded:

1. **DenoiseNet** — a compact residual CNN (DnCNN-style) that learns to remove Gaussian noise.
2. **SRNet** — a compact sub-pixel CNN (ESPCN-style) that learns 2× super-resolution.

Both are trained from scratch on BSDS300, exported to **ONNX**, and designed to be dropped straight into
the RestoreAI Flask app's `weights/` folder — the app already uses OpenCV's DNN module for inference, so
no new production dependency (e.g. PyTorch) is needed at deploy time.

**Before running:** `Runtime → Change runtime type → T4 GPU` (or any available GPU). CPU will also work, just slower.

**What you need:** `images.zip` (containing `train/` and `test/` folders of BSDS300 images) uploaded to your
Google Drive. Update `DRIVE_ZIP_PATH` in the next cell to match where you put it.


In [ ]:
# Mount Google Drive and unzip the dataset
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

# >>> EDIT THIS to the exact path of your uploaded zip in Google Drive <<<
DRIVE_ZIP_PATH = "/content/drive/MyDrive/restoreai/images.zip"

EXTRACT_DIR = "/content/bsds300"
if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

# Locate the train/ and test/ folders wherever the zip put them
def find_dir(root, name):
    for dirpath, dirnames, _ in os.walk(root):
        if name in dirnames:
            return os.path.join(dirpath, name)
    raise FileNotFoundError(f"Could not find a '{name}' folder under {root}")

TRAIN_DIR = find_dir(EXTRACT_DIR, "train")
TEST_DIR = find_dir(EXTRACT_DIR, "test")
print("Train dir:", TRAIN_DIR, "-", len(os.listdir(TRAIN_DIR)), "files")
print("Test dir: ", TEST_DIR, "-", len(os.listdir(TEST_DIR)), "files")


In [ ]:
# Imports & setup
import random, time, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def list_images(folder):
    exts = (".jpg", ".jpeg", ".png")
    return sorted([os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(exts)])

train_files = list_images(TRAIN_DIR)
test_files = list_images(TEST_DIR)

# Hold out the last 20 training images as a validation set
val_files = train_files[-20:]
train_files = train_files[:-20]
print(f"{len(train_files)} train / {len(val_files)} val / {len(test_files)} test images")


## Dataset

One `PatchDataset` class, parameterized by `task` (`"denoise"` or `"sr"`), so both models share the same data pipeline. Patches are cropped randomly and flipped/rotated for cheap augmentation — useful since BSDS300 is a small dataset (300 images total).

In [ ]:
PATCH = 64        # HR / clean patch size used for both tasks
SCALE = 2          # super-resolution factor (matches the app's "Standard" quality tier)
NOISE_SIGMA_RANGE = (10, 40)   # pixel-value noise std, out of 255

class PatchDataset(Dataset):
    # task=\"denoise\": returns (noisy_patch, clean_patch), both PATCHxPATCH, 3xHxW, float in [0,1]
    # task=\"sr\":      returns (lr_patch, hr_patch) where lr_patch is PATCH/SCALE and hr_patch is PATCH
    def __init__(self, files, task, patch=PATCH, scale=SCALE, patches_per_image=8, augment=True):
        self.files = files
        self.task = task
        self.patch = patch
        self.scale = scale
        self.patches_per_image = patches_per_image
        self.augment = augment

    def __len__(self):
        return len(self.files) * self.patches_per_image

    def _augment(self, arr):
        if not self.augment:
            return arr
        if random.random() < 0.5:
            arr = np.fliplr(arr)
        if random.random() < 0.5:
            arr = np.flipud(arr)
        k = random.choice([0, 1, 2, 3])
        arr = np.rot90(arr, k)
        return np.ascontiguousarray(arr)

    def __getitem__(self, idx):
        file_idx = idx % len(self.files)
        img = Image.open(self.files[file_idx]).convert("RGB")
        w, h = img.size
        p = self.patch
        if w < p or h < p:
            img = img.resize((max(w, p), max(h, p)))
            w, h = img.size
        x = random.randint(0, w - p)
        y = random.randint(0, h - p)
        patch = img.crop((x, y, x + p, y + p))
        arr = np.asarray(patch, dtype=np.float32) / 255.0  # HxWx3 in [0,1]
        arr = self._augment(arr)

        if self.task == "denoise":
            clean = arr.copy()
            sigma = random.uniform(*NOISE_SIGMA_RANGE) / 255.0
            noisy = clean + np.random.normal(0, sigma, clean.shape).astype(np.float32)
            noisy = np.clip(noisy, 0.0, 1.0)
            x_t = torch.from_numpy(noisy.transpose(2, 0, 1).copy())
            y_t = torch.from_numpy(clean.transpose(2, 0, 1).copy())
            return x_t, y_t

        elif self.task == "sr":
            hr = arr.copy()
            hr_img = Image.fromarray((hr * 255).astype(np.uint8))
            lr_img = hr_img.resize((p // self.scale, p // self.scale), Image.BICUBIC)
            lr = np.asarray(lr_img, dtype=np.float32) / 255.0
            x_t = torch.from_numpy(lr.transpose(2, 0, 1).copy())
            y_t = torch.from_numpy(hr.transpose(2, 0, 1).copy())
            return x_t, y_t

        else:
            raise ValueError(f"Unknown task: {self.task}")


## Model architectures

Both are small and fully convolutional (no fixed-size layers), so the exported ONNX models can run on full-resolution images of any size at inference time — not just the 64×64 training patches.

In [ ]:
class DenoiseNet(nn.Module):
    # Compact DnCNN-style residual denoiser: predicts the noise, subtracts it from the input.
    def __init__(self, channels=3, features=48, num_layers=8):
        super().__init__()
        layers = [nn.Conv2d(channels, features, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(num_layers - 2):
            layers += [
                nn.Conv2d(features, features, 3, padding=1),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True),
            ]
        layers += [nn.Conv2d(features, channels, 3, padding=1)]
        self.body = nn.Sequential(*layers)

    def forward(self, x):
        predicted_noise = self.body(x)
        return torch.clamp(x - predicted_noise, 0.0, 1.0)


class SRNet(nn.Module):
    # Compact ESPCN-style sub-pixel CNN for fixed-factor super-resolution.
    def __init__(self, scale=SCALE, channels=3, d=64, s=32):
        super().__init__()
        self.scale = scale
        self.feature = nn.Sequential(
            nn.Conv2d(channels, d, 5, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(d, s, 3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(s, s, 3, padding=1), nn.ReLU(inplace=True),
        )
        self.upsample = nn.Sequential(
            nn.Conv2d(s, channels * (scale ** 2), 3, padding=1),
            nn.PixelShuffle(scale),
        )

    def forward(self, x):
        return torch.clamp(self.upsample(self.feature(x)), 0.0, 1.0)

print("DenoiseNet params:", sum(p.numel() for p in DenoiseNet().parameters()))
print("SRNet params:     ", sum(p.numel() for p in SRNet().parameters()))


## Training utilities

In [ ]:
def psnr(pred, target):
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return 99.0
    return 10 * math.log10(1.0 / mse)


def train_model(model, train_loader, val_loader, epochs, lr=1e-3, name="model"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.L1Loss()

    history = {"train_loss": [], "val_psnr": []}
    best_psnr = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
        train_loss = running_loss / len(train_loader.dataset)
        scheduler.step()

        model.eval()
        val_psnr_total, n = 0.0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                val_psnr_total += psnr(out, y) * x.size(0)
                n += x.size(0)
        val_psnr = val_psnr_total / n

        history["train_loss"].append(train_loss)
        history["val_psnr"].append(val_psnr)
        print(f"[{name}] epoch {epoch:02d}/{epochs}  loss={train_loss:.4f}  val_psnr={val_psnr:.2f} dB")

        if val_psnr > best_psnr:
            best_psnr = val_psnr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    print(f"[{name}] best val PSNR: {best_psnr:.2f} dB")
    return model, history


def plot_history(history, name):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
    ax[0].plot(history["train_loss"]); ax[0].set_title(f"{name} — train loss"); ax[0].set_xlabel("epoch")
    ax[1].plot(history["val_psnr"]); ax[1].set_title(f"{name} — val PSNR (dB)"); ax[1].set_xlabel("epoch")
    plt.tight_layout(); plt.show()


## Train the denoising model

~25–35 epochs is usually enough on a dataset this size — adjust `EPOCHS` if you have more Colab GPU time available.

In [ ]:
EPOCHS_DENOISE = 30
BATCH_SIZE = 32

train_ds_dn = PatchDataset(train_files, task="denoise", patches_per_image=10, augment=True)
val_ds_dn = PatchDataset(val_files, task="denoise", patches_per_image=4, augment=False)

train_loader_dn = DataLoader(train_ds_dn, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader_dn = DataLoader(val_ds_dn, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

denoise_model = DenoiseNet()
t0 = time.time()
denoise_model, dn_history = train_model(denoise_model, train_loader_dn, val_loader_dn,
                                         epochs=EPOCHS_DENOISE, lr=1e-3, name="DenoiseNet")
print(f"Training took {time.time()-t0:.1f}s")
plot_history(dn_history, "DenoiseNet")


### Evaluate denoising on the held-out test set

Compares noisy-vs-clean PSNR against restored-vs-clean PSNR — the real, measured improvement your trained model provides.

In [ ]:
def evaluate_denoise_on_testset(model, files, sigma=25, n_samples=None):
    model.eval()
    files_to_use = files if n_samples is None else files[:n_samples]
    noisy_psnrs, restored_psnrs = [], []
    with torch.no_grad():
        for f in files_to_use:
            img = Image.open(f).convert("RGB")
            arr = np.asarray(img, dtype=np.float32) / 255.0
            noisy = np.clip(arr + np.random.normal(0, sigma / 255.0, arr.shape), 0, 1).astype(np.float32)

            clean_t = torch.from_numpy(arr.transpose(2, 0, 1)).unsqueeze(0).to(device)
            noisy_t = torch.from_numpy(noisy.transpose(2, 0, 1)).unsqueeze(0).to(device)
            out_t = model(noisy_t)

            noisy_psnrs.append(psnr(noisy_t, clean_t))
            restored_psnrs.append(psnr(out_t, clean_t))
    return float(np.mean(noisy_psnrs)), float(np.mean(restored_psnrs))

noisy_avg, restored_avg = evaluate_denoise_on_testset(denoise_model, test_files, sigma=25)
print(f"Average PSNR — noisy input: {noisy_avg:.2f} dB  |  after DenoiseNet: {restored_avg:.2f} dB  "
      f"(+{restored_avg - noisy_avg:.2f} dB)")


### Export DenoiseNet to ONNX

In [ ]:
denoise_model.eval()
dummy = torch.randn(1, 3, 128, 128).to(device)

torch.onnx.export(
    denoise_model, dummy, "denoise_model.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {2: "height", 3: "width"}, "output": {2: "height", 3: "width"}},
    opset_version=11,
)

# Some PyTorch versions write weights to a companion <name>.onnx.data file
# instead of embedding them. Reload (picking up that external data if present)
# and re-save as a single self-contained file, so the one .onnx you download
# is complete on its own.
import onnx
m = onnx.load("denoise_model.onnx", load_external_data=True)
onnx.save_model(m, "denoise_model.onnx", save_as_external_data=False)
print("Exported self-contained denoise_model.onnx")

from google.colab import files as colab_files
colab_files.download("denoise_model.onnx")


## Train the super-resolution model (2×)

In [ ]:
EPOCHS_SR = 40

train_ds_sr = PatchDataset(train_files, task="sr", patches_per_image=10, augment=True)
val_ds_sr = PatchDataset(val_files, task="sr", patches_per_image=4, augment=False)

train_loader_sr = DataLoader(train_ds_sr, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
val_loader_sr = DataLoader(val_ds_sr, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

sr_model = SRNet(scale=SCALE)
t0 = time.time()
sr_model, sr_history = train_model(sr_model, train_loader_sr, val_loader_sr,
                                    epochs=EPOCHS_SR, lr=1e-3, name="SRNet")
print(f"Training took {time.time()-t0:.1f}s")
plot_history(sr_history, "SRNet")


### Evaluate super-resolution on the held-out test set

Compares plain bicubic upscaling against the trained model, both against the true high-resolution image.

In [ ]:
def evaluate_sr_on_testset(model, files, scale=SCALE, n_samples=None):
    model.eval()
    files_to_use = files if n_samples is None else files[:n_samples]
    bicubic_psnrs, model_psnrs = [], []
    with torch.no_grad():
        for f in files_to_use:
            hr_img = Image.open(f).convert("RGB")
            w, h = hr_img.size
            w, h = (w // scale) * scale, (h // scale) * scale
            hr_img = hr_img.resize((w, h))
            lr_img = hr_img.resize((w // scale, h // scale), Image.BICUBIC)
            bicubic_up = lr_img.resize((w, h), Image.BICUBIC)

            hr_arr = np.asarray(hr_img, dtype=np.float32) / 255.0
            bicubic_arr = np.asarray(bicubic_up, dtype=np.float32) / 255.0
            lr_arr = np.asarray(lr_img, dtype=np.float32) / 255.0

            hr_t = torch.from_numpy(hr_arr.transpose(2, 0, 1)).unsqueeze(0).to(device)
            bicubic_t = torch.from_numpy(bicubic_arr.transpose(2, 0, 1)).unsqueeze(0).to(device)
            lr_t = torch.from_numpy(lr_arr.transpose(2, 0, 1)).unsqueeze(0).to(device)

            out_t = model(lr_t)

            bicubic_psnrs.append(psnr(bicubic_t, hr_t))
            model_psnrs.append(psnr(out_t, hr_t))
    return float(np.mean(bicubic_psnrs)), float(np.mean(model_psnrs))

bicubic_avg, model_avg = evaluate_sr_on_testset(sr_model, test_files, scale=SCALE)
print(f"Average PSNR — bicubic upscale: {bicubic_avg:.2f} dB  |  after SRNet: {model_avg:.2f} dB  "
      f"(+{model_avg - bicubic_avg:.2f} dB)")


### Export SRNet to ONNX

In [ ]:
sr_model.eval()
dummy_lr = torch.randn(1, 3, 64, 64).to(device)

torch.onnx.export(
    sr_model, dummy_lr, "sr_model_x2.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {2: "height", 3: "width"}, "output": {2: "height", 3: "width"}},
    opset_version=11,
)

import onnx
m = onnx.load("sr_model_x2.onnx", load_external_data=True)
onnx.save_model(m, "sr_model_x2.onnx", save_as_external_data=False)
print("Exported self-contained sr_model_x2.onnx")

colab_files.download("sr_model_x2.onnx")


## Next step: integrating these into RestoreAI

Once `denoise_model.onnx` and `sr_model_x2.onnx` have downloaded to your computer:

1. Copy both files into the Flask app's `weights/` folder.
2. Send them to me (or just tell me they're ready) and I'll wire them into
   `models/restoration_model.py` and `models/super_resolution.py` as the new default
   denoising/super-resolution backends, loaded via `cv2.dnn.readNetFromONNX(...)` —
   the same OpenCV DNN approach already used for the pretrained ESPCN model, so no new
   production dependency is introduced.
3. The existing classical methods (Non-Local Means denoising, bicubic-based ESPCN) stay
   in the codebase as a fallback / comparison point — useful for your report's "trained
   model vs. classical baseline" section.

For reference, here's the inference-side pattern these models will use in production:

```python
net = cv2.dnn.readNetFromONNX("weights/denoise_model.onnx")
blob = cv2.dnn.blobFromImage(img_bgr, scalefactor=1/255.0, swapRB=True)
net.setInput(blob)
out = net.forward()               # 1 x 3 x H x W, float32 in [0,1]
out_img = (out[0].transpose(1, 2, 0) * 255).clip(0, 255).astype('uint8')
```
